# Experimento 2: closed-book frente a RAG

**Pregunta experimental.** Con el mismo LLM, las mismas preguntas y el mismo decoding, ¿qué cambia en la corrección de la respuesta, en las citas y en el respaldo de las afirmaciones cuando el modelo recibe los 3 primeros artículos de la condición C del experimento 1?

Fijos: `Qwen/Qwen2.5-0.5B-Instruct` (el mismo de la PC1 y la PC2), decoding greedy, `max_new_tokens = 220`, 10 consultas del benchmark elegidas antes de ejecutar y 2 controles negativos sin respuesta en el Código Civil, donde lo correcto es abstenerse.

Variable modificada: con o sin evidencia en el prompt.

Interfaz de salida (Semana 3): JSON con `respuesta`, `citas` y `abstencion`, validado con JSON Schema. Las citas tienen patrón fijo y no hay campos con enum de texto libre: en la PC2, una tilde (`prohibición_permiso` frente a `prohibicion_permiso`) hizo inválidas 4 de 6 salidas.

## 0. Entorno

In [1]:
!python -m pip install -q --break-system-packages -r ../requirements.txt

In [2]:
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

import config, corpus, metricas
print("raíz:", ROOT)
print("RUN_REAL_RETRIEVAL:", config.RUN_REAL_RETRIEVAL, "| RUN_REAL_RERANKER:", config.RUN_REAL_RERANKER,
      "| RUN_REAL_LLM:", config.RUN_REAL_LLM)

raíz: /workspace/CC-0F4/Evaluaciones/parcial
RUN_REAL_RETRIEVAL: True | RUN_REAL_RERANKER: True | RUN_REAL_LLM: True


In [3]:
import rag
from recuperacion import BM25, DenseRetriever, Reranker, run_condition

passages = corpus.load_passages()
queries, qrels, references = corpus.load_benchmark()
corpus.validate_benchmark(passages, queries, qrels, references)
query_text = {q["query_id"]: q["query"] for q in queries}
dense, bm25, reranker = DenseRetriever(passages), BM25(passages), Reranker()
print("consultas RAG:", [r["query_id"] for r in references])

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 401.67it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (526 > 512). Running this sequence through the model will result in indexing errors
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 389.11it/s]


consultas RAG: ['C01', 'C02', 'C03', 'C05', 'C06', 'C10', 'C11', 'C17', 'C19', 'C20', 'N01', 'N02']


In [4]:
gen = rag.Generator()
print("generador:", gen.model_id)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 396.63it/s]


generador: Qwen/Qwen2.5-0.5B-Instruct


## 1. Evidencia de las Semanas 1 y 2 sobre el modelo usado

Qwen2.5 es un decoder causal: cada token solo atiende a los anteriores (máscara causal), y la atención usa más cabezas de consulta que de clave y valor (grouped-query attention), lo que reduce el tamaño del KV cache.

In [5]:
if gen.real:
    c = gen.model.config
    arch = {"capas": c.num_hidden_layers, "dim_oculta": c.hidden_size, "cabezas_Q": c.num_attention_heads,
            "cabezas_KV": c.num_key_value_heads, "vocabulario": c.vocab_size,
            "posiciones_max": c.max_position_embeddings,
            "parametros_M": round(sum(p.numel() for p in gen.model.parameters()) / 1e6, 1)}
    print(arch)
    ejemplo = query_text["C05"]
    toks = gen.tokenizer.tokenize(ejemplo)
    print(f"{len(toks)} tokens Qwen:", toks)
    print(f"{len(dense.model.tokenizer.tokenize(ejemplo)) if dense.real else 'sin E5'} tokens E5 para la misma consulta")
else:
    print("Sin generador real: esta celda necesita RUN_REAL_LLM=1.")

{'capas': 24, 'dim_oculta': 896, 'cabezas_Q': 14, 'cabezas_KV': 2, 'vocabulario': 151936, 'posiciones_max': 32768, 'parametros_M': 494.0}
34 tokens Qwen: ['Si', 'Ġel', 'ĠÃºltimo', 'ĠdÃŃa', 'Ġde', 'Ġun', 'Ġpl', 'azo', 'Ġde', 'Ġcad', 'uc', 'idad', 'Ġca', 'e', 'Ġen', 'ĠdÃŃa', 'Ġinh', 'Ã¡', 'bil', ',', 'ĠÂ¿', 'el', 'Ġpl', 'azo', 'Ġse', 'Ġext', 'i', 'ende', 'Ġal', 'Ġsiguiente', 'ĠdÃŃa', 'ĠhÃ¡', 'bil', '?']
32 tokens E5 para la misma consulta


## 2. Evidencia recuperada (condición C, top-3)

In [6]:
items = []
for ref in references:
    qid = ref["query_id"]
    ranking, _ = run_condition(config.RAG_CONDITION, ref["query"], dense, bm25, reranker)
    ctx = ranking[:config.RAG_TOP_K]
    ctx_ids = [c["passage_id"] for c in ctx]
    gold = ref["evidencia_esperada"]
    items.append({"query_id": qid, "query": ref["query"], "control_negativo": ref["control_negativo"],
                  "ranking": ranking, "contexto": ctx_ids,
                  "recall_contexto": metricas.recall_at_k(ctx_ids, gold, config.RAG_TOP_K) if gold else None})
pd.DataFrame([{k: v for k, v in it.items() if k != "ranking"} for it in items])

,query_id,query,control_negativo,contexto,recall_contexto
0,C01,¿A partir de qué edad una persona tiene plena capacidad de ejercicio?,False,"[cc_art_42, cc_art_44, cc_art_46]",1.000000
1,C02,¿Desde qué edad el matrimonio hace cesar la incapacidad de un menor?,False,"[cc_art_46, cc_art_537, cc_art_549]",0.500000
2,C03,¿En qué plazo puede un asociado impugnar judicialmente un acuerdo de la asociación?,False,"[cc_art_92, cc_art_96, cc_art_85]",1.000000
3,C05,"Si el último día de un plazo de caducidad cae en día inhábil, ¿el plazo se extiende al siguiente día hábil?",False,"[cc_art_2007, cc_art_183, cc_art_2002]",1.000000
4,C06,¿En cuántos años prescribe la acción de nulidad del acto jurídico?,False,"[cc_art_2001, cc_art_537, cc_art_450]",1.000000
5,C10,"Vivo en una casa desde hace diez años como si fuera mía, ¿puedo convertirme en propietario?",False,"[cc_art_950, cc_art_920, cc_art_533]",1.000000
6,C11,¿Qué parte de la herencia corresponde como legítima a los hijos?,False,"[cc_art_725, cc_art_729, cc_art_723]",0.666667
7,C17,"El contrato fija una penalidad desproporcionada por incumplimiento, ¿el juez puede rebajarla?",False,"[cc_art_1346, cc_art_1533, cc_art_1341]",1.000000
8,C19,"Compré una casa y descubrí un defecto oculto, ¿cuánto tiempo tengo para reclamar al vendedor?",False,"[cc_art_1523, cc_art_1576, cc_art_1753]",0.000000
9,C20,¿Cuál es el plazo máximo de un arrendamiento de duración determinada?,False,"[cc_art_1688, cc_art_1699, cc_art_1689]",1.000000


## 3. Generación

Para cada consulta se generan dos respuestas con el mismo modelo y el mismo decoding: sin evidencia y con evidencia. Se guardan la salida cruda, la validación y la latencia.

In [12]:
# Cada respuesta se escribe en disco apenas se genera. Si la celda se interrumpe,
# al volver a ejecutarla continúa desde la primera respuesta que falte.
out_path = Path("results/exp2_respuestas.jsonl")
done = {}
if out_path.exists():
    for line in out_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            o = json.loads(line)
            done[(o["query_id"], o["condicion"])] = o

outputs = []
with out_path.open("a", encoding="utf-8") as fh:
    for it in items:
        for cond in ["closed_book", "rag"]:
            key = (it["query_id"], cond)
            if key in done:
                outputs.append(done[key])
                print(f"{key[0]} {cond:11s} ya estaba guardada")
                continue
            print(f"{time.strftime('%H:%M:%S')} generando {key[0]} {cond}...", flush=True)
            ranking = it["ranking"] if cond == "rag" else None
            ctx_ids = it["contexto"] if cond == "rag" else []
            messages = rag.build_messages(it["query"], ranking)
            raw, ms = gen.generate(messages)
            v = rag.validate_output(raw, ctx_ids)
            ref = next(r for r in references if r["query_id"] == it["query_id"])
            gold = set(ref["evidencia_esperada"])
            o = {
                "query_id": it["query_id"], "condicion": cond, "control_negativo": it["control_negativo"],
                "pregunta": it["query"], "contexto": ctx_ids, "prompt_tokens": gen.prompt_tokens(messages),
                "latencia_ms": round(ms), "salida_cruda": raw, **v,
                "citas_relevantes": [c for c in v["citas"] if c in gold],
                "token_f1_proxy": rag.token_f1_proxy(v["respuesta"] or "", ref["respuesta_referencia"]),
            }
            fh.write(json.dumps(o, ensure_ascii=False) + "\n")
            fh.flush()
            outputs.append(o)
            print(f"   json={v['json_valido']!s:5} schema={v['schema_valido']!s:5} "
                  f"citas={v['citas']} abst={v['abstencion']} {ms/1000:.1f}s", flush=True)
print("respuestas:", len(outputs))

18:11:09 generando C01 closed_book...
   json=True  schema=True  citas=[] abst=False 5.9s
18:11:15 generando C01 rag...
   json=True  schema=True  citas=[] abst=False 12.6s
18:11:27 generando C02 closed_book...
   json=True  schema=True  citas=[] abst=False 3.7s
18:11:31 generando C02 rag...
   json=True  schema=True  citas=[] abst=False 13.0s
18:11:44 generando C03 closed_book...
   json=True  schema=True  citas=[] abst=False 4.2s
18:11:48 generando C03 rag...
   json=True  schema=True  citas=[] abst=False 13.2s
18:12:02 generando C05 closed_book...
   json=True  schema=True  citas=[] abst=False 5.6s
18:12:07 generando C05 rag...
   json=True  schema=True  citas=[] abst=False 9.5s
18:12:17 generando C06 closed_book...
   json=True  schema=True  citas=[] abst=False 4.1s
18:12:21 generando C06 rag...
   json=True  schema=True  citas=[] abst=False 11.7s
18:12:33 generando C10 closed_book...
   json=True  schema=False citas=['La propiedad es de otra persona'] abst=False 4.6s
18:12:37 gene

## 4. Logits, softmax y ventana de contexto (Semana 2)

Con greedy, el token elegido es el de mayor probabilidad después del softmax. El prompt RAG es mucho más largo: el prefill procesa más tokens y el KV cache guarda claves y valores de todos ellos.

In [8]:
if gen.real:
    it = items[0]
    for cond, ranking in [("closed_book", None), ("rag", it["ranking"])]:
        msgs = rag.build_messages(it["query"], ranking)
        print(cond, "| tokens del prompt:", gen.prompt_tokens(msgs), "| top-5 del primer token:", gen.next_token_distribution(msgs))
    tok_table = pd.DataFrame(outputs).groupby("condicion").prompt_tokens.agg(["mean", "max"]).round(0)
    display(tok_table)
else:
    print("Sin generador real: esta celda necesita RUN_REAL_LLM=1.")

closed_book | tokens del prompt: 172 | top-5 del primer token: [('{"', 0.9212), ('{\n', 0.0459), ('```', 0.0315), ("{'", 0.0007), ('{', 0.0004)]
rag | tokens del prompt: 790 | top-5 del primer token: [('{"', 0.7628), ('{\n', 0.1929), ('```', 0.043), ('{', 0.0005), ("{'", 0.0004)]


,mean,max
condicion,,
closed_book,179.0,191
rag,602.0,790


## 5. Verificaciones automáticas

Estas columnas solo comprueban forma y procedencia: si la salida es JSON, si cumple el schema, si las citas están entre los artículos entregados y si coinciden con la evidencia esperada. `token_f1_proxy` mide solapamiento de palabras con la referencia y no se usa como prueba de corrección.

In [9]:
out = pd.DataFrame(outputs)
pos = out[~out.control_negativo]
neg = out[out.control_negativo]
auto = pd.DataFrame({
    "json_valido": out.groupby("condicion").json_valido.mean(),
    "schema_valido": out.groupby("condicion").schema_valido.mean(),
    "con_citas": pos.groupby("condicion").citas.apply(lambda s: float(np.mean([len(c) > 0 for c in s]))),
    "citas_fuera_de_contexto": out.groupby("condicion").citas_fuera_de_contexto.apply(lambda s: int(sum(len(c) for c in s))),
    "citas_relevantes_sobre_citas": pos.groupby("condicion").apply(
        lambda g: sum(len(x) for x in g.citas_relevantes) / max(sum(len(x) for x in g.citas), 1), include_groups=False),
    "abstencion_en_positivas": pos.groupby("condicion").abstencion.apply(lambda s: float(np.mean([a is True for a in s]))),
    "abstencion_en_negativas": neg.groupby("condicion").abstencion.apply(lambda s: float(np.mean([a is True for a in s]))),
    "token_f1_proxy": pos.groupby("condicion").token_f1_proxy.mean(),
    "latencia_ms": out.groupby("condicion").latencia_ms.mean(),
}).round(3)
auto.to_csv("results/exp2_verificacion_automatica.csv")
auto

,json_valido,schema_valido,con_citas,citas_fuera_de_contexto,citas_relevantes_sobre_citas,abstencion_en_positivas,abstencion_en_negativas,token_f1_proxy,latencia_ms
condicion,,,,,,,,,
closed_book,1.0,0.667,0.4,4,0.0,0.0,0.0,0.074,14894.833
rag,1.0,1.000,0.0,0,0.0,0.1,0.5,0.077,34497.333


In [10]:
cols = ["query_id", "condicion", "respuesta", "citas", "abstencion", "schema_valido", "citas_fuera_de_contexto", "errores_schema"]
out[cols]

,query_id,condicion,respuesta,citas,abstencion,schema_valido,citas_fuera_de_contexto,errores_schema
0,C01,closed_book,18 años,[],False,True,[],[]
1,C01,rag,a partir de 16 años,[],False,True,[],[]
2,C02,closed_book,18 años,[],False,True,[],[]
3,C02,rag,más de dieciséis (16) años,[],False,True,[],[]
4,C03,closed_book,30 días,[],False,True,[],[]
5,C03,rag,no hay evidencia suficiente,[],False,True,[],[]
6,C05,closed_book,El plazo se extiende al siguiente día hábil,[],False,True,[],[]
7,C05,rag,true,[],False,True,[],[]
8,C06,closed_book,10,[],False,True,[],[]
9,C06,rag,tres,[],False,True,[],[]


## 6. Auditoría manual

La corrección, la corrección de citas y el respaldo (faithfulness) se juzgan a mano en `results/auditoria_manual_rag.csv`, una fila por afirmación atómica, con el criterio del Cuaderno 7: una afirmación está respaldada solo si un artículo del contexto entregado la contiene de forma explícita. En closed-book no hay contexto, así que ninguna afirmación puede estar respaldada por evidencia entregada.

In [13]:
audit_path = Path("results/auditoria_manual_rag.csv")
if audit_path.exists():
    audit = pd.read_csv(audit_path)
    claims = audit.groupby("condicion").agg(afirmaciones=("respaldada", "size"), respaldadas=("respaldada", "sum"))
    claims["faithfulness"] = (claims.respaldadas / claims.afirmaciones).round(3)
    per_answer = audit.drop_duplicates(["query_id", "condicion"])
    answers = per_answer.groupby("condicion").agg(respuesta_correcta=("respuesta_correcta", "mean"),
                                                  citas_correctas=("citas_correctas", "mean")).round(3)
    manual = answers.join(claims)
    manual.to_csv("results/exp2_auditoria_resumen.csv")
    display(manual)
else:
    print("No existe results/auditoria_manual_rag.csv.")

,respuesta_correcta,citas_correctas,afirmaciones,respaldadas,faithfulness
condicion,,,,,
closed_book,0.167,NaN,12,0,0.00
rag,0.208,0.083,12,3,0.25
